![imagenes](logo.png)

# K-medias (K-Means Clustering)

Imaginemos que tenemos un conjunto de puntos distribuidos en el espacio, pero sin etiquetas: no sabemos a qué grupo pertenece cada observación.
El objetivo del algoritmo K-medias es descubrir esas agrupaciones ocultas dividiendo los puntos en 
𝐾
K grupos (clusters), de modo que los puntos dentro de un mismo grupo sean lo más parecidos posible entre sí, y lo más distintos posible de los de otros grupos.

La idea central es geométrica:
cada grupo se representa por un centroide, y cada punto pertenece al centroide más cercano.
Después, los centroides se reajustan iterativamente hasta que dejan de moverse.

En el espacio, esto equivale a rellenar el plano con regiones de Voronoi:
cada región contiene los puntos que están más próximos a su centroide.
Así, el algoritmo busca el equilibrio entre la compacidad interna de los grupos y la separación entre ellos.

<img src="im022.png" width="50%" style="display: block; margin-left: auto; margin-right: auto;">

## Idea matemática y función objetivo

El criterio que optimiza K-medias es la **inercia intra-cluster** (suma de cuadrados dentro de los grupos). Si $$ C_1, \ldots, C_K $$ son los grupos y $ \mu_k $ su centroide, el objetivo es  

$$\min_{\{C_k\}, \{\mu_k\}} \sum_{k=1}^K \sum_{x \in C_k} \|x - \mu_k\|^2.$$

Dos movimientos alternan hasta converger:  

1. **Asignación**: cada $x$ va al $\mathrm{argmin}_k\|x-\mu_k\|$.  
2. **Actualización**: $\mu_k \leftarrow$ promedio de los puntos de $C_k$.  

La frontera entre grupos está formada por **bisectrices**: son líneas (o hiperplanos) que separan regiones de Voronoi. El resultado final es una partición **poligonal** del plano.

<img src="im023.png" width="50%" style="display: block; margin-left: auto; margin-right: auto;">

## Algoritmo de K-medias (paso a paso)

Partimos de $K$ centroides iniciales (al azar o propuestos). Luego alternamos dos pasos:

1. **Asignación.** Para cada punto $x$, elegimos el grupo más cercano:  
$$c(x) = \mathrm{argmin}_{k=1,\ldots,K} \|x - \mu_k\|$$

2. **Actualización.** Recalculamos cada centroide como el promedio de su grupo:  
$$\mu_k \leftarrow \frac{1}{|C_k|} \sum_{x \in C_k} x$$

El proceso desciende la **inercia intra-cluster** y se detiene al estabilizar asignaciones o centroides. Diferentes inicializaciones pueden llevar a mínimos distintos.

<img src="im024.png" width="100%" style="display: block; margin-left: auto; margin-right: auto;">

## k-means++ (inicialización inteligente)

La calidad del resultado depende mucho de los puntos iniciales. **k-means++** elige los centroides iniciales de forma **probabilística** favoreciendo la **dispersión**:

1. Elegimos un punto al azar como primer centroide.
2. Para cada punto $x$, calculamos $D(x)^2$: el cuadrado de su distancia al centroide más cercano elegido hasta ahora.
3. Elegimos el siguiente centroide con probabilidad proporcional a $D(x)^2$.
4. Repetimos hasta tener $K$ centroides y luego ejecutamos K-medias normal.

Este procedimiento tiende a colocar los centroides lejos entre sí y reduce la probabilidad de caer en malos mínimos.

<img src="im025.png" width="100%" style="display: block; margin-left: auto; margin-right: auto;">

## Elección de K — Método del codo

Aumentar $K$ siempre **reduce** la inercia (SSE), pero a partir de cierto punto las mejoras son **marginales**. El método del **codo busca el $K$ donde la caída de la inercia cambia de ritmo**: antes del codo, cada nuevo cluster aporta mucho; después, aporta poco.

<img src="im026.png" width="60%" style="display: block; margin-left: auto; margin-right: auto;">

## Elección de K — Coeficiente de silueta

La **silueta** mide cuán bien queda un punto dentro de su cluster frente a otros: para cada punto $i$,  

- $a(i) = \text{distancia media a puntos de su propio cluster},$  
- $b(i) = \text{mínima distancia media a los puntos de otro cluster},$ y  

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} \in [-1, 1].$$

Valores cercanos a 1 indican asignaciones limpias; cercanos a 0, fronteras difusas; negativos, mala asignación. Se suele elegir el $K$ con silueta media más alta y perfiles sin colas problemáticas.

<img src="im027.png" width="80%" style="display: block; margin-left: auto; margin-right: auto;">

## Límites de K-medias y buenas prácticas

K-medias modela clusters **compactos** y **aproximadamente esféricos** bajo distancia euclídea. Por eso falla cuando la geometría real es **no convexa** (dos lunas, anillos), cuando hay **varianzas muy distintas** entre grupos, **tamaños desiguales o outliers** que arrastran los centroides. Además, no es adecuado para **variables categóricas puras** (la media carece de sentido) ni para datos con escalas heterogéneas si no se **normaliza/estandariza antes**.

Buenas prácticas mínimas:

- Escalado previo (p. ej. StandardScaler o MinMaxScaler).
- Inicialización robusta (k-means++, varias corridas n_init) y semilla para reproducibilidad.
- Elección de K con codo/silueta pero validando con criterio de negocio o etiquetas externas cuando existan.
- Considerar **alternativas** si la forma no es esférica: **DBSCAN**, **HDBSCAN**, **Spectral Clustering**, **GMM** (si las nubes son elípticas), o transformar el espacio (kernelización, embeddings) antes de agrupar.

<img src="im028.png" width="90%" style="display: block; margin-left: auto; margin-right: auto;">